In [1]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import importlib
import gradio as gr 
import pandas as pd
import utility_function
importlib.reload(utility_function)
from utility_function import(
    generate_search_queries,
    search_all_queries, 
    deduplicate_publications, 
    filter_relevant_publications,
    process_relevant_publications, 
    save_json
)

In [2]:
#Load environment variables in a file called .env
#Print the key prefixes to help with any debugging
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
ollama_api_key = os.getenv('OLLAMA_API_KEY')

#Check each key
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key and anthropic_api_key.startswith("sk-or-"):
    print("Anthropic API Key exists")
else:
    print("Anthropic API Key not set")

if ollama_api_key:
    print("Ollama API KEY exist")
else:
    print("Ollama API Key do not exist")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists
Ollama API KEY exist


In [3]:
#Load model URLs from .env
ANTHROPIC_BASE_URL = os.getenv('ANTHROPIC_BASE_URL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')

#Check each URL
if ANTHROPIC_BASE_URL:
    print(f"Anthropic base URL exist as {ANTHROPIC_BASE_URL}")
else:
    print("Anthropic base URL do not exist")

if OLLAMA_BASE_URL:
    print(f"Ollama base URL exist as {OLLAMA_BASE_URL}")
else:
    print("Ollama base ULR do not exist")

Anthropic base URL exist as https://openrouter.ai/api/v1
Ollama base URL exist as http://localhost:11434/v1


In [4]:
#Create two different OpenA1 Python clients. Connect tAnthropic and Ollama
openrouter = OpenAI(
    base_url=ANTHROPIC_BASE_URL,
    api_key=anthropic_api_key,
    timeout=120.0,
)

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=ollama_api_key,
    timeout=120.0,
)

In [5]:
QUERY_GENERATION_MODEL = "anthropic/claude-sonnet-5"
PUBLICATION_EXTRACTION_MODEL = {
    "model": "openai/gpt-4.1-mini",
    "reasoning": {
        "effort": "medium"
    },
    "temperature": 0
}

### Main Pipeline 

In [6]:
def run_literature_mining_pipeline(
    topic,
    maximum_search_queries=5,
    results_per_query=10,
    maximum_candidates_sent_to_llm=None,
    relevance_threshold=0.70,
    from_year=None,
    until_year=None,
    output_directory="data/outputs",
):
    """
    Run the complete Automated Literature Mining Pipeline.

    Workflow
    --------
    1. Model 1 generates focused search queries.
    2. Python searches Crossref.
    3. Python normalizes and deduplicates candidates.
    4. Model 1 classifies semantic relevance.
    5. Python retrieves publisher article pages.
    6. Python extracts deterministic metadata and article text.
    7. Model 2 extracts study objectives and main findings.
    8. Python validates and saves the results.
    """
    started_at = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    print("=" * 70)
    print("AUTOMATED LITERATURE MINING PIPELINE")
    print("=" * 70)
    print(f"Topic: {topic}")

    # ---------------------------------------------------------
    # Step 1: Generate search strategy
    # ---------------------------------------------------------
    print("\n[1/7] Generating search queries...")

    search_strategy = generate_search_queries(
        topic=topic,
        maximum_queries=maximum_search_queries,
    )

    for query in search_strategy["search_queries"]:
        print(f"  - {query}")

    # ---------------------------------------------------------
    # Step 2: Discover candidate publications
    # ---------------------------------------------------------
    print("\n[2/7] Searching scholarly metadata...")

    search_results = search_all_queries(
        search_queries=search_strategy[
            "search_queries"
        ],
        results_per_query=results_per_query,
        from_year=from_year,
        until_year=until_year,
    )

    raw_candidates = search_results["candidates"]

    print(
        f"Raw candidate records: "
        f"{len(raw_candidates)}"
    )

    # ---------------------------------------------------------
    # Step 3: Deduplicate
    # ---------------------------------------------------------
    print("\n[3/7] Deduplicating publications...")

    unique_candidates = deduplicate_publications(
        raw_candidates
    )

    print(
        f"Unique candidate publications: "
        f"{len(unique_candidates)}"
    )

    candidates_for_classification = (
        unique_candidates[
            :maximum_candidates_sent_to_llm
        ]
    )

    # ---------------------------------------------------------
    # Step 4: Relevance classification
    # ---------------------------------------------------------
    print("\n[4/7] Classifying relevance...")

    relevance_results = (
        filter_relevant_publications(
            topic=topic,
            candidates=candidates_for_classification,
            relevance_threshold=relevance_threshold,
        )
    )

    relevant_publications = relevance_results[
        "relevant"
    ]

    print(
        f"Relevant publications: "
        f"{len(relevant_publications)}"
    )
    print(
        f"Rejected publications: "
        f"{len(relevance_results['rejected'])}"
    )
    print(
        f"Failed classifications: "
        f"{len(relevance_results['failed'])}"
    )

    # ---------------------------------------------------------
    # Step 5 and 6: Retrieve and extract
    # ---------------------------------------------------------
    print(
        "\n[5/7] Retrieving publisher pages and "
        "extracting article content..."
    )

    extraction_results = (
        process_relevant_publications(
            publications=relevant_publications,
            delay_seconds=1.0,
        )
    )

    successful_publications = extraction_results[
        "successful"
    ]

    print(
        f"Successfully extracted publications: "
        f"{len(successful_publications)}"
    )
    print(
        f"Failed article extractions: "
        f"{len(extraction_results['failed'])}"
    )

    # ---------------------------------------------------------
    # Step 7: Build final output
    # ---------------------------------------------------------
    print("\n[6/7] Building final result...")

    final_result = {
        "pipeline": {
            "name": (
                "Automated Literature Mining Pipeline"
            ),
            "started_at": started_at,
            "completed_at": time.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "topic": topic,
            "discovery_model": QUERY_GENERATION_MODEL,
            "extraction_model": PUBLICATION_EXTRACTION_MODEL["model"],
            "relevance_threshold": (
                relevance_threshold
            ),
            "from_year": from_year,
            "until_year": until_year,
        },
        "summary": {
            "queries_generated": len(
                search_strategy["search_queries"]
            ),
            "raw_candidates": len(raw_candidates),
            "unique_candidates": len(
                unique_candidates
            ),
            "classified_candidates": len(
                candidates_for_classification
            ),
            "relevant_publications": len(
                relevant_publications
            ),
            "successful_extractions": len(
                successful_publications
            ),
            "failed_extractions": len(
                extraction_results["failed"]
            ),
        },
        "search_strategy": search_strategy,
        "publications": successful_publications,
        "rejected_publications": (
            relevance_results["rejected"]
        ),
        "failed_queries": search_results[
            "failed_queries"
        ],
        "failed_classifications": (
            relevance_results["failed"]
        ),
        "failed_extractions": extraction_results[
            "failed"
        ],
    }

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------
    print("\n[7/7] Saving outputs...")

    output_directory = Path(output_directory)

    json_path = save_json(
        data=final_result,
        output_path=(
            output_directory
            / "literature_mining_results.json"
        ),
    )

    print(f"JSON output: {json_path}")

    print("\nPipeline completed.")

    return final_result


In [7]:
# -------------------------------------------------------------
# Gradio-only helper
# -------------------------------------------------------------
def format_gradio_outputs(result):
    """
    Convert the pipeline result into Gradio outputs.
    """

    summary = result.get(
        "summary",
        {}
    )

    relevant_publications = result.get(
        "publications",
        []
    )

    failed_extractions = result.get(
        "failed_extractions",
        []
    )

    relevant_table = pd.DataFrame(
        relevant_publications
    )

    failed_table = pd.DataFrame(
        failed_extractions
    )

    json_path = (
        "data/outputs/"
        "literature_mining_results.json"
    )

    return (
        result,
        summary,
        relevant_table,
        failed_table,
        json_path,
        result,
    )

def run_from_gradio(
    topic,
    from_year,
    until_year,
    maximum_search_queries,
    results_per_query,
    maximum_candidates_sent_to_llm,
    relevance_threshold
):
    #Stream initial status
    yield(
        "### Running\nStarting literature mining pipeline...",
        {},
        {},
        pd.DataFrame(),
        pd.DataFrame(),
        None,
        {}
    )

    result = run_literature_mining_pipeline(
        topic=topic,
        maximum_search_queries=int(
            maximum_search_queries
        ),
        results_per_query=int(
            results_per_query
        ),
        maximum_candidates_sent_to_llm=(
            int(maximum_candidates_sent_to_llm)
            if maximum_candidates_sent_to_llm is not None
            else None
        ),
        relevance_threshold=float(
            relevance_threshold
        ),
        from_year=(
            int(from_year)
            if from_year is not None
            else None
        ),
        until_year=(
            int(until_year)
            if until_year is not None
            else None
        ),
        output_directory="data/outputs",
    )

    outputs = format_gradio_outputs(
        result
    )

    yield(
        "### Completed\nPipeline completed successfully.",
        *outputs
    )

################################# Rerun function has been deleted ###########################################################
def clear_gradio_outputs():
    """
    Reset Gradio inputs and outputs.
    """
    return (
        (
            "Long-read sequencing for structural variant "
            "analysis in breast cancer"
        ),
        None,
        None,
        5,
        10,
        None,
        0.70,
        {},
        {},
        pd.DataFrame(),
        pd.DataFrame(),
        None,
        None,
        {},
    )

# -------------------------------------------------------------
# Gradio styling
# -------------------------------------------------------------
custom_css = """
/* Main app container */
.gradio-container {
    max-width: 1400px !important;
    margin: 0 auto !important;
}

/* Main title */
#main-title {
    text-align: center;
    font-size: 38px !important;
    font-weight: 700;
    color: #163A5F;
    margin-bottom: 4px;
}

/* Subtitle */
#main-subtitle {
    text-align: center;
    font-size: 17px !important;
    color: #5B6573;
    margin-bottom: 25px;
}

/* Section headings */
.section-title {
    font-size: 23px !important;
    font-weight: 650;
    color: #163A5F;
    margin-top: 15px;
    margin-bottom: 8px;
}

/* Status box */
#pipeline-status {
    font-size: 16px !important;
    padding: 12px 16px;
    border-radius: 10px;
    border: 1px solid #D8DEE8;
    margin-bottom: 15px;
}

/* Input labels */
label {
    font-size: 15px !important;
    font-weight: 600 !important;
}

/* Textbox */
textarea {
    font-size: 16px !important;
}

/* Number inputs */
input {
    font-size: 15px !important;
}

/* Buttons */
button {
    font-size: 15px !important;
    font-weight: 600 !important;
}

/* Primary Run button */
#run-button {
    font-size: 16px !important;
    font-weight: 700 !important;
}

/* Tabs */
button[role="tab"] {
    font-size: 15px !important;
    font-weight: 600 !important;
}

/* Dataframe */
.gr-dataframe {
    font-size: 14px !important;
}

/* JSON */
.gr-json {
    font-size: 14px !important;
}
"""

# -------------------------------------------------------------
# Base theme
# -------------------------------------------------------------

app_theme = gr.themes.Soft(
    primary_hue="sky",
    secondary_hue="slate",
)

# -------------------------------------------------------------
# Gradio Blocks application
# -------------------------------------------------------------
with gr.Blocks(
    title="Automated Literature Mining Pipeline",
    theme=app_theme,
    css=custom_css
) as view:

    stored_result = gr.State(
        value={}
    )

    # ---------------------------------------------------------
    # Application title
    # ---------------------------------------------------------

    gr.HTML(
        """
        <div id="main-title">
            Automated Literature Mining Pipeline
        </div>

        <div id="main-subtitle">
            Discover, classify, retrieve, and extract
            scientific publications for a research topic.
        </div>
        """
    )

    # ---------------------------------------------------------
    # Pipeline status
    # ---------------------------------------------------------

    status_output = gr.Markdown(
        "### Ready\nConfigure your search and run the pipeline.",
        elem_id="pipeline-status",
    )

    # ---------------------------------------------------------
    # Search configuration
    # ---------------------------------------------------------

    gr.HTML(
        """
        <div class="section-title">
            Search Configuration
        </div>
        """
    )

    topic_input = gr.Textbox(
        label="Research Topic",
        value=(
            "Long-read sequencing for structural variant "
            "analysis in breast cancer"
        ),
        lines=3,
        placeholder=(
            "Enter a focused scientific research topic"
        ),
    )

    with gr.Row():
        from_year_input = gr.Number(
            value=None,
            precision=0,
            label="From Year",
        )

        until_year_input = gr.Number(
            value=None,
            precision=0,
            label="Until Year",
        )

    # ---------------------------------------------------------
    # Advanced settings
    # ---------------------------------------------------------
    with gr.Accordion(
        "Advanced Settings",
        open=False,
    ):
        maximum_search_queries_input = gr.Number(
            value=5,
            precision=0,
            label="Maximum Search Queries",
        )

        results_per_query_input = gr.Number(
            value=10,
            precision=0,
            label="Results Per Query",
        )

        maximum_candidates_input = gr.Number(
            value=None,
            precision=0,
            label="Maximum Candidates Sent to LLM",
            info=(
                "Leave blank to classify all unique "
                "candidates."
            ),
        )

        relevance_threshold_input = gr.Slider(
            minimum=0.0,
            maximum=1.0,
            value=0.70,
            step=0.01,
            label="Relevance Threshold",
        )

    # ---------------------------------------------------------
    # Buttons
    # ---------------------------------------------------------
    with gr.Row():
        run_button = gr.Button(
            "Run Pipeline",
            variant="primary",
        )

        cancel_button = gr.Button(
            "Cancel",
            variant="stop",
        )

        clear_button = gr.Button(
            "Clear",
        )

    # ---------------------------------------------------------
    # Results
    # ---------------------------------------------------------
    gr.HTML(
        """
        <div class="section-title">
            Pipeline Results
        </div>
        """
    )
    
    with gr.Tabs():

        with gr.Tab("Complete Result"):
            complete_result_output = gr.JSON(
                label="Complete Pipeline Result"
            )

        with gr.Tab("Summary"):
            summary_output = gr.JSON(
                label="Pipeline Summary"
            )

        with gr.Tab("Relevant Publications"):
            relevant_publications_output = (
                gr.Dataframe(
                    label="Relevant Publications",
                    interactive=False,
                    wrap=True,
                )
            )

        with gr.Tab("Failed Extractions"):
            failed_extractions_output = (
                gr.Dataframe(
                    label="Failed Extractions",
                    interactive=False,
                    wrap=True,
                )
            )

        with gr.Tab("Downloads"):
            json_download_output = (
                gr.DownloadButton(
                    label="Download JSON"
                )
            )

    # ---------------------------------------------------------
    # Run event
    # ---------------------------------------------------------
    run_event = run_button.click(
        fn=run_from_gradio,
        inputs=[
            topic_input,
            from_year_input,
            until_year_input,
            maximum_search_queries_input,
            results_per_query_input,
            maximum_candidates_input,
            relevance_threshold_input,
        ],
        outputs=[
            status_output,
            complete_result_output,
            summary_output,
            relevant_publications_output,
            failed_extractions_output,
            json_download_output,
            stored_result,
        ],
    )

    # ---------------------------------------------------------
    # Cancel
    # ---------------------------------------------------------
    cancel_button.click(
        fn=None,
        cancels=[
            run_event,
        ],
    )

    # ---------------------------------------------------------
    # Clear
    # ---------------------------------------------------------
    clear_button.click(
        fn=clear_gradio_outputs,
        inputs=[],
        outputs=[
            topic_input,
            from_year_input,
            until_year_input,
            maximum_search_queries_input,
            results_per_query_input,
            maximum_candidates_input,
            relevance_threshold_input,
            complete_result_output,
            summary_output,
            relevant_publications_output,
            failed_extractions_output,
            json_download_output,
            stored_result,
        ],
    )

if __name__ == "__main__":
    view.queue()
    view.launch(
        share=True,
        inbrowser=True, 
        #auth=("sadjei65320", 
        #"DonKutus@4141")
        )

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9414c6f99b217b9c01.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


gio: https://9414c6f99b217b9c01.gradio.live: Operation not supported


AUTOMATED LITERATURE MINING PIPELINE
Topic: Long-read sequencing for structural variant analysis in breast cancer

[1/7] Generating search queries...


Traceback (most recent call last):
  File "/home/sadjei65320/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 849, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sadjei65320/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sadjei65320/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sadjei65320/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sadjei65320/Projects/llm